# Explore UBL OASIS Google Drive Folder

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/setup-ubl-sheets-access-zWGFj/notebooks/explore-drive-folder.ipynb)

Recursively explores the OASIS UBL TC Google Drive folder and discovers
**all** files and their revision history using **every available API
method**. Saves results to Google Drive for later analysis.

## What this notebook does

1. **Lists every file** in the OASIS UBL TC shared folder (recursively)
2. **Fetches revision history** via Drive API v3 `revisions.list`
3. **Probes internal max revision** via binary search on export URLs
4. **Fetches revision history** via Drive API v2 `revisions.list`
   (different fields: `exportLinks`, `pinned`, `published`)
5. **Fetches `files.get` metadata** via both v3 and v2 (`version`,
   `headRevisionId`, `exportLinks`)
6. **Cross-compares** all four methods to identify sheets with deep
   editing history
7. **Spot-checks** non-archived sheets by downloading sample revisions
8. **Saves a complete inventory** as JSON to Google Drive

## API Methods Used

| # | Method | Endpoint | What it provides |
|---|--------|----------|-----------------|
| 1 | **Drive v3 `revisions.list`** | `drive/v3/.../revisions` | Timestamps, authors, sizes |
| 2 | **Binary search probe** | `docs.google.com/.../export` | True internal max revision |
| 3 | **Drive v2 `revisions.list`** | `drive/v2/.../revisions` | `exportLinks`, `pinned`, `published` |
| 4 | **Drive v3/v2 `files.get`** | `drive/v3/files/{id}` | `version`, `headRevisionId` |

## Known sheets (already explored)

| Sheet | ID | Revisions | Status |
|-------|----|-----------|--------|
| UBL 2.5 Library | `18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY` | ~2005 | Fully archived |
| UBL 2.5 Documents | `1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg` | ~2204 | Fully archived |
| UBL 2.5 Signature | `1T6z2NZ4mc69YllZOXE5TnT5Ey-FlVtaXN1oQ4AIMp7g` | ? | Not yet explored |
| UBL 2.4 Library | `1kxlFLz2thJOlvpq2ChRAcv76SiKgEIRtoVRqsZ7OBUs` | ? | Not yet explored |
| UBL 2.4 Documents | `1GNpHCS7_QkJtP3QIOdPJWL5N3kQ1EzPznT6M8sPsA0Y` | ? | Not yet explored |

## Output

```
Drive: ubl-gc-revisions/
├── drive-discovery.json              ← complete folder tree + all revision data
├── drive-discovery-summary.txt       ← human-readable summary
└── (existing revision archives...)
```

In [ ]:
# === Step 0: Auth + Mount Drive ===
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

import google.auth
from google.auth.transport.requests import Request as AuthRequest
from datetime import datetime, timezone

creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive.readonly']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')
if creds.expiry:
    remaining = (creds.expiry - datetime.now(timezone.utc).replace(tzinfo=None)).total_seconds()
    print(f'Token expiry: {creds.expiry.isoformat()} ({remaining:.0f}s from now)')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive output: {DRIVE_DIR}')

## Step 1: Configuration & Helpers

In [ ]:
import json, time, sys, hashlib
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from collections import Counter

# Root folder: OASIS UBL TC shared Google Drive folder
ROOT_FOLDER_ID = '0B4X4evii3UjcdG5wNlVFTXlaYVU'

# Already-explored sheet IDs (for tagging in output)
KNOWN_SHEETS = {
    '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY': 'ubl25_library (ARCHIVED)',
    '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg': 'ubl25_documents (ARCHIVED)',
    '1T6z2NZ4mc69YllZOXE5TnT5Ey-FlVtaXN1oQ4AIMp7g': 'ubl25_signature (KNOWN)',
    '1kxlFLz2thJOlvpq2ChRAcv76SiKgEIRtoVRqsZ7OBUs': 'ubl24_library (KNOWN)',
    '1GNpHCS7_QkJtP3QIOdPJWL5N3kQ1EzPznT6M8sPsA0Y': 'ubl24_documents (KNOWN)',
}

# Known max revisions from prior binary search (for verification)
KNOWN_MAX_REVS = {
    '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY': 2005,
    '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg': 2204,
}

# Rate limiting
API_DELAY = 0.3  # seconds between API calls

# Provenance tracking
api_log = []


def now_iso():
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')


def ensure_fresh_token():
    """Refresh TOKEN if near expiry."""
    global TOKEN
    if not creds.expiry:
        return
    remaining = (creds.expiry - datetime.now(timezone.utc).replace(tzinfo=None)).total_seconds()
    if remaining > 300:
        return
    old = TOKEN[:8]
    creds.token = None
    creds.refresh(AuthRequest())
    TOKEN = creds.token
    print(f'  >> Token refreshed: {old}... -> {TOKEN[:8]}...')


def api_get(url, context=''):
    """Authenticated GET with retry, rate limiting, and provenance logging."""
    ensure_fresh_token()
    headers = {'Authorization': f'Bearer {TOKEN}'}
    for attempt in range(4):
        call_time = now_iso()
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=30) as resp:
                data = json.loads(resp.read())
            api_log.append({
                'timestamp': call_time, 'url': url,
                'status': 200, 'attempt': attempt + 1, 'context': context,
            })
            time.sleep(API_DELAY)
            return data
        except HTTPError as e:
            api_log.append({
                'timestamp': call_time, 'url': url,
                'status': e.code, 'attempt': attempt + 1, 'context': context,
            })
            if e.code in (429, 500, 502, 503) and attempt < 3:
                wait = 2 ** (attempt + 1)
                print(f'    [{e.code}] retrying in {wait}s...')
                time.sleep(wait)
                continue
            if e.code != 404:
                print(f'    ERROR {e.code}: {context}')
            return None
        except Exception as exc:
            api_log.append({
                'timestamp': call_time, 'url': url,
                'status': 0, 'attempt': attempt + 1, 'context': context,
                'error': str(exc),
            })
            if attempt < 3:
                time.sleep(2 ** (attempt + 1))
                continue
            print(f'    EXCEPTION: {exc}')
            return None
    return None


def api_head(url, context=''):
    """Authenticated HEAD request — for binary search probing."""
    ensure_fresh_token()
    headers = {'Authorization': f'Bearer {TOKEN}'}
    call_time = now_iso()
    try:
        req = Request(url, headers=headers, method='HEAD')
        with urlopen(req, timeout=15) as resp:
            api_log.append({
                'timestamp': call_time, 'url': url,
                'status': resp.status, 'attempt': 1, 'context': context,
            })
            return resp.status == 200
    except HTTPError as e:
        api_log.append({
            'timestamp': call_time, 'url': url,
            'status': e.code, 'attempt': 1, 'context': context,
        })
        return False
    except Exception:
        return False


print('Helpers ready')

## Step 2: Recursive Folder Exploration

Walks the entire OASIS UBL TC Google Drive folder tree. For every
Google Sheets spreadsheet found, fetches its complete revision history
via the Drive API v3 Revisions endpoint.

In [ ]:
def list_folder(folder_id):
    """List all files in a folder (paginated)."""
    all_files = []
    page_token = None
    page = 0
    while True:
        page += 1
        url = (
            f'https://www.googleapis.com/drive/v3/files'
            f'?q=%27{folder_id}%27+in+parents+and+trashed%3Dfalse'
            f'&fields=nextPageToken,files(id,name,mimeType,modifiedTime,'
            f'createdTime,size,owners/displayName,owners/emailAddress,'
            f'lastModifyingUser/displayName,lastModifyingUser/emailAddress,'
            f'shared,webViewLink)'
            f'&pageSize=100&orderBy=name'
        )
        if page_token:
            url += f'&pageToken={page_token}'
        data = api_get(url, f'list:{folder_id}:p{page}')
        if not data:
            break
        all_files.extend(data.get('files', []))
        page_token = data.get('nextPageToken')
        if not page_token:
            break
    return all_files


def get_revisions_v3(file_id, file_name):
    """Fetch all revisions of a file via Drive API v3 (paginated).

    Requests all available v3 fields including keepForever, published,
    publishAuto, publishedOutsideDomain, and publishedLink.
    """
    all_revs = []
    page_token = None
    page = 0
    while True:
        page += 1
        url = (
            f'https://www.googleapis.com/drive/v3/files/{file_id}/revisions'
            f'?pageSize=1000'
            f'&fields=nextPageToken,revisions(id,modifiedTime,'
            f'lastModifyingUser/displayName,lastModifyingUser/emailAddress,'
            f'size,exportLinks,keepForever,published,publishAuto,'
            f'publishedOutsideDomain,publishedLink)'
        )
        if page_token:
            url += f'&pageToken={page_token}'
        data = api_get(url, f'v3_revisions:{file_id}:{file_name}:p{page}')
        if not data:
            break
        all_revs.extend(data.get('revisions', []))
        page_token = data.get('nextPageToken')
        if not page_token:
            break
    return all_revs


# Keep backward-compatible alias used by explore_folder
get_revisions = get_revisions_v3


def explore_folder(folder_id, folder_name, path='/', depth=0):
    """Recursively explore a folder and fetch revision history for spreadsheets."""
    indent = '  ' * depth
    print(f'{indent}[folder] {folder_name}/')
    
    files = list_folder(folder_id)
    print(f'{indent}  {len(files)} items')
    
    result = {
        'id': folder_id,
        'name': folder_name,
        'path': path,
        'type': 'folder',
        'children': [],
    }
    
    for f in files:
        mime = f.get('mimeType', '')
        name = f.get('name', '?')
        fid = f.get('id', '')
        child_path = f'{path}{name}'
        
        entry = {
            'id': fid,
            'name': name,
            'path': child_path,
            'mimeType': mime,
            'modifiedTime': f.get('modifiedTime'),
            'createdTime': f.get('createdTime'),
            'size': f.get('size'),
            'owners': f.get('owners'),
            'lastModifyingUser': f.get('lastModifyingUser'),
            'webViewLink': f.get('webViewLink'),
        }
        
        if mime == 'application/vnd.google-apps.folder':
            sub = explore_folder(fid, name, f'{child_path}/', depth + 1)
            entry['children'] = sub['children']
            entry['type'] = 'folder'
        
        elif mime == 'application/vnd.google-apps.spreadsheet':
            entry['type'] = 'spreadsheet'
            known_tag = KNOWN_SHEETS.get(fid)
            if known_tag:
                entry['known_as'] = known_tag
            
            revisions = get_revisions(fid, name)
            entry['revision_count'] = len(revisions)
            entry['revisions'] = revisions
            
            if revisions:
                entry['first_revision_time'] = revisions[0].get('modifiedTime')
                entry['last_revision_time'] = revisions[-1].get('modifiedTime')
                authors = set()
                for rev in revisions:
                    user = rev.get('lastModifyingUser', {})
                    dn = user.get('displayName') or user.get('emailAddress')
                    if dn:
                        authors.add(dn)
                entry['unique_authors'] = sorted(authors)
            
            tag = f' [{known_tag}]' if known_tag else ''
            print(f'{indent}  [sheet] {name}: {len(revisions)} revisions{tag}')
        
        elif mime == 'application/vnd.google-apps.shortcut':
            entry['type'] = 'shortcut'
            target_url = (
                f'https://www.googleapis.com/drive/v3/files/{fid}'
                f'?fields=shortcutDetails(targetId,targetMimeType,targetResourceKey)'
            )
            target_data = api_get(target_url, f'shortcut:{fid}:{name}')
            if target_data and 'shortcutDetails' in target_data:
                entry['shortcutTarget'] = target_data['shortcutDetails']
                tid = target_data['shortcutDetails'].get('targetId', '?')
                tmime = target_data['shortcutDetails'].get('targetMimeType', '?')
                print(f'{indent}  [shortcut] {name} -> {tid[:12]}... ({tmime})')
                
                # Follow shortcuts to spreadsheets
                if tmime == 'application/vnd.google-apps.spreadsheet':
                    revisions = get_revisions(tid, f'(shortcut) {name}')
                    entry['shortcutTarget']['revision_count'] = len(revisions)
                    entry['shortcutTarget']['revisions'] = revisions
                    if revisions:
                        entry['shortcutTarget']['first_revision_time'] = revisions[0].get('modifiedTime')
                        entry['shortcutTarget']['last_revision_time'] = revisions[-1].get('modifiedTime')
                    known_tag = KNOWN_SHEETS.get(tid)
                    if known_tag:
                        entry['shortcutTarget']['known_as'] = known_tag
                    tag = f' [{known_tag}]' if known_tag else ''
                    print(f'{indent}    -> {len(revisions)} revisions{tag}')
        
        else:
            entry['type'] = 'file'
            size_str = f' ({int(f.get("size", 0)):,}b)' if f.get('size') else ''
            print(f'{indent}  [file] {name}{size_str}')
        
        result['children'].append(entry)
    
    return result


print('Exploration functions ready')

In [ ]:
# === Run the exploration ===
start_time = now_iso()
print(f'Started: {start_time}')
print(f'Root folder: {ROOT_FOLDER_ID}')
print(f'URL: https://drive.google.com/drive/folders/{ROOT_FOLDER_ID}')
print()

tree = explore_folder(ROOT_FOLDER_ID, 'UBL TC (root)')

end_time = now_iso()
print(f'\nCompleted: {end_time}')
print(f'API calls: {len(api_log)}')

## Step 3: Analyze Results

Extract all spreadsheets, count their revisions, and identify which
ones have significant revision history that hasn't been explored yet.

In [ ]:
def collect_spreadsheets(node, results=None):
    """Recursively collect all spreadsheet entries from the tree."""
    if results is None:
        results = []
    
    for child in node.get('children', []):
        t = child.get('type')
        if t == 'spreadsheet':
            results.append(child)
        elif t == 'shortcut':
            target = child.get('shortcutTarget', {})
            if target.get('revision_count', 0) > 0:
                # Create a synthetic entry for the shortcut target
                results.append({
                    'id': target.get('targetId'),
                    'name': f"(shortcut) {child['name']}",
                    'path': child.get('path', ''),
                    'type': 'spreadsheet',
                    'revision_count': target.get('revision_count', 0),
                    'first_revision_time': target.get('first_revision_time'),
                    'last_revision_time': target.get('last_revision_time'),
                    'known_as': target.get('known_as'),
                    'via_shortcut': True,
                    'revisions': target.get('revisions', []),
                })
        elif t == 'folder':
            collect_spreadsheets(child, results)
    
    return results


def count_all(node):
    """Count totals in the tree."""
    folders = files = sheets = shortcuts = revisions = 0
    for child in node.get('children', []):
        t = child.get('type')
        if t == 'folder':
            folders += 1
            f2, fi2, s2, sc2, r2 = count_all(child)
            folders += f2; files += fi2; sheets += s2
            shortcuts += sc2; revisions += r2
        elif t == 'spreadsheet':
            sheets += 1
            revisions += child.get('revision_count', 0)
        elif t == 'shortcut':
            shortcuts += 1
            revisions += child.get('shortcutTarget', {}).get('revision_count', 0)
        else:
            files += 1
    return folders, files, sheets, shortcuts, revisions


# Collect all spreadsheets
all_sheets = collect_spreadsheets(tree)
folders, files, sheets, shortcuts, total_revs = count_all(tree)

print('=' * 70)
print('FOLDER INVENTORY')
print('=' * 70)
print(f'  Folders:       {folders}')
print(f'  Files:         {files}')
print(f'  Spreadsheets:  {sheets}')
print(f'  Shortcuts:     {shortcuts}')
print(f'  Total revisions (Drive API): {total_revs}')
print()

# Sort by revision count (most revisions first)
all_sheets.sort(key=lambda s: s.get('revision_count', 0), reverse=True)

print('=' * 70)
print('ALL SPREADSHEETS (by revision count)')
print('=' * 70)
print()

already_archived_ids = {
    '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
    '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
}

new_discoveries = []

for i, s in enumerate(all_sheets, 1):
    fid = s.get('id', '?')
    name = s.get('name', '?')
    rc = s.get('revision_count', 0)
    first = (s.get('first_revision_time') or '?')[:10]
    last = (s.get('last_revision_time') or '?')[:10]
    known = s.get('known_as', '')
    path = s.get('path', '')
    
    status = ''
    if fid in already_archived_ids:
        status = ' [ARCHIVED]'
    elif known:
        status = f' [{known}]'
    elif rc > 1:
        status = ' *** NEW ***'
        new_discoveries.append(s)
    
    shortcut_tag = ' (via shortcut)' if s.get('via_shortcut') else ''
    
    print(f'{i:3d}. {name}')
    print(f'     ID: {fid}')
    print(f'     Path: {path}')
    print(f'     Revisions: {rc} ({first} -> {last}){status}{shortcut_tag}')
    
    # Show authors for sheets with significant history
    if rc > 5:
        authors = set()
        for rev in s.get('revisions', []):
            user = rev.get('lastModifyingUser', {})
            dn = user.get('displayName') or user.get('emailAddress')
            if dn:
                authors.add(dn)
        if authors:
            print(f'     Authors: {", ".join(sorted(authors))}')
    print()

print('=' * 70)
print(f'NEW DISCOVERIES (sheets with >1 revision, not already known)')
print('=' * 70)

if new_discoveries:
    for s in new_discoveries:
        rc = s.get('revision_count', 0)
        name = s.get('name', '?')
        fid = s.get('id', '?')
        first = (s.get('first_revision_time') or '?')[:10]
        last = (s.get('last_revision_time') or '?')[:10]
        print(f'  {name}')
        print(f'    ID: {fid}, {rc} revisions ({first} -> {last})')
else:
    print('  (none found)')

print()

## Step 4: Detailed Revision Timeline

For each spreadsheet with significant revision history (>5 revisions),
show a timeline of edits with timestamps and authors.

In [ ]:
# Show detailed revision timeline for sheets with significant history
DETAIL_THRESHOLD = 5  # show details for sheets with more than this many revisions

significant_sheets = [
    s for s in all_sheets
    if s.get('revision_count', 0) > DETAIL_THRESHOLD
]

print(f'Sheets with >{DETAIL_THRESHOLD} revisions: {len(significant_sheets)}')
print()

for s in significant_sheets:
    fid = s.get('id', '?')
    name = s.get('name', '?')
    revisions = s.get('revisions', [])
    known = s.get('known_as', '')
    tag = f' [{known}]' if known else ''
    
    print(f'={"="*68}')
    print(f'{name}{tag}')
    print(f'  ID: {fid}')
    print(f'  Revisions: {len(revisions)}')
    print(f'={"="*68}')
    
    # Show first 10, last 10, and any large gaps
    if len(revisions) <= 30:
        show_revs = revisions
    else:
        show_revs = revisions[:10]
        show_revs.append({'_separator': True, '_count': len(revisions) - 20})
        show_revs.extend(revisions[-10:])
    
    for j, rev in enumerate(show_revs):
        if rev.get('_separator'):
            print(f'  ... ({rev["_count"]} more revisions) ...')
            continue
        
        rev_id = rev.get('id', '?')
        mod_time = rev.get('modifiedTime', '?')
        user = rev.get('lastModifyingUser', {})
        author = user.get('displayName') or user.get('emailAddress', '?')
        size = rev.get('size', '?')
        
        print(f'  rev {rev_id:>5s}  {mod_time[:19]}  {author:30s}  {size}')
    
    # Activity analysis: edits per month
    print(f'\n  Monthly activity:')
    months = {}
    for rev in revisions:
        mt = rev.get('modifiedTime', '')
        if mt and len(mt) >= 7:
            month = mt[:7]
            months[month] = months.get(month, 0) + 1
    
    for month in sorted(months.keys()):
        bar = '#' * min(months[month], 60)
        print(f'    {month}: {months[month]:4d} {bar}')
    
    print()

## Step 5: Internal Revision Count Probe

The Drive API `revisions.list` returns a limited set of revisions
(typically 25-100). The actual **internal revision counter** can be much
higher (the UBL 2.5 Library sheet has 2005+ internal revisions despite
the API listing only ~25).

For each non-archived spreadsheet with significant history, probe for
the true max revision number by testing export URLs with binary search.

In [ ]:
def probe_max_revision(sheet_id, start_guess=100, max_probe=10000):
    """Binary search for the highest valid internal revision number.
    
    Google Sheets' internal revision counter is independent of the
    Drive API revision list. The export URL accepts any valid revision
    number and returns 400/404 for invalid ones.
    
    Returns (max_rev, probes_used).
    """
    ensure_fresh_token()
    headers = {'Authorization': f'Bearer {TOKEN}'}
    probes = 0
    
    def test_rev(n):
        nonlocal probes
        probes += 1
        url = (
            f'https://docs.google.com/spreadsheets/export'
            f'?id={sheet_id}&revision={n}&exportFormat=csv'
        )
        try:
            req = Request(url, headers=headers, method='HEAD')
            with urlopen(req, timeout=15) as resp:
                return resp.status == 200
        except HTTPError:
            return False
        except Exception:
            return False
    
    # Phase 1: Find upper bound by doubling
    upper = start_guess
    while upper <= max_probe:
        if test_rev(upper):
            upper *= 2
            time.sleep(0.5)
        else:
            break
        time.sleep(0.5)
    
    if upper > max_probe:
        return max_probe, probes  # hit ceiling
    
    # Phase 2: Binary search between upper/2 and upper
    lo = max(1, upper // 2)
    hi = upper
    
    while lo < hi - 1:
        mid = (lo + hi) // 2
        if test_rev(mid):
            lo = mid
        else:
            hi = mid
        time.sleep(0.5)
    
    return lo, probes


# Probe non-archived sheets
sheets_to_probe = [
    s for s in all_sheets
    if s.get('revision_count', 0) > 0
    and s.get('id') not in already_archived_ids
    and s.get('type') == 'spreadsheet'
]

print(f'Probing {len(sheets_to_probe)} sheets for true max revision...\n')

probe_results = []

for s in sheets_to_probe:
    fid = s.get('id', '?')
    name = s.get('name', '?')
    api_rev_count = s.get('revision_count', 0)
    known = s.get('known_as', '')
    tag = f' [{known}]' if known else ''
    
    print(f'  {name}{tag} (API says {api_rev_count} revisions)...', end=' ', flush=True)
    
    max_rev, probes = probe_max_revision(fid)
    ratio = max_rev / api_rev_count if api_rev_count > 0 else 0
    
    result = {
        'id': fid,
        'name': name,
        'api_revision_count': api_rev_count,
        'internal_max_revision': max_rev,
        'ratio': round(ratio, 1),
        'probes_used': probes,
        'known_as': known,
    }
    probe_results.append(result)
    
    interesting = ' *** INTERESTING ***' if max_rev > 50 else ''
    print(f'max_rev={max_rev} (ratio={ratio:.1f}x, {probes} probes){interesting}')
    time.sleep(1)  # be gentle

# Summary
print(f'\n{"="*70}')
print(f'INTERNAL REVISION PROBE RESULTS')
print(f'{"="*70}')

probe_results.sort(key=lambda r: r['internal_max_revision'], reverse=True)

for r in probe_results:
    tag = f' [{r["known_as"]}]' if r['known_as'] else ''
    flag = ' <<<' if r['internal_max_revision'] > 50 else ''
    print(f'  {r["name"]:45s} API: {r["api_revision_count"]:5d}  '
          f'Internal: {r["internal_max_revision"]:6d}  '
          f'Ratio: {r["ratio"]:6.1f}x{tag}{flag}')

## Step 6: Drive API v2 `revisions.list`

The v2 API often returns **different revisions** than v3, and includes
fields that v3 dropped:
- `exportLinks` — direct download URLs per format (revision-specific!)
- `pinned` — whether a revision was manually pinned
- `published` / `publishAuto` — publication state
- `selfLink` — canonical revision URL

The `exportLinks` are particularly valuable: they return revision-specific
content (unlike v3 export, which ignores the revision parameter).

In [ ]:
def get_revisions_v2(file_id, label):
    """Fetch all revisions via Drive API v2 (paginated).
    
    v2 uses 'items' (not 'revisions'), 'modifiedDate' (not 'modifiedTime'),
    and includes exportLinks, pinned, published, selfLink, etc.
    """
    all_revs = []
    page_token = None
    page = 0
    while True:
        page += 1
        url = (
            f'https://www.googleapis.com/drive/v2/files/{file_id}/revisions'
            f'?maxResults=1000'
        )
        if page_token:
            url += f'&pageToken={page_token}'
        data = api_get(url, f'v2_revisions:{label}:p{page}')
        if not data:
            break
        all_revs.extend(data.get('items', []))
        page_token = data.get('nextPageToken')
        if not page_token:
            break
    return all_revs


# Fetch v2 revisions for every spreadsheet found in the folder exploration.
# We focus on sheets with >0 v3 revisions (i.e. we have access).
v2_results = {}

print('=' * 70)
print('DRIVE API v2 revisions.list — all discovered spreadsheets')
print('=' * 70)
print()

for s in all_sheets:
    fid = s.get('id', '?')
    name = s.get('name', '?')
    v3_count = s.get('revision_count', 0)
    known = s.get('known_as', '')
    tag = f' [{known}]' if known else ''

    if v3_count == 0:
        continue

    print(f'{name}{tag} (v3={v3_count})...', end=' ', flush=True)
    revs = get_revisions_v2(fid, name)
    v2_results[fid] = revs

    if revs:
        pinned = [r for r in revs if r.get('pinned')]
        with_export = [r for r in revs if r.get('exportLinks')]
        first_time = revs[0].get('modifiedDate', '?')[:19]
        last_time = revs[-1].get('modifiedDate', '?')[:19]
        print(f'{len(revs)} revisions  (pinned={len(pinned)}, '
              f'exportLinks={len(with_export)}/{len(revs)})')
        print(f'  Time: {first_time} -> {last_time}')
    else:
        print('NONE')
    time.sleep(0.5)

print(f'\nTotal sheets queried: {len(v2_results)}')

In [ ]:
# Detailed v2 revision listing — show v2-specific fields
for s in all_sheets:
    fid = s.get('id', '?')
    name = s.get('name', '?')
    v2_revs = v2_results.get(fid, [])
    if not v2_revs:
        continue

    known = s.get('known_as', '')
    tag = f' [{known}]' if known else ''

    print(f'\n{"=" * 70}')
    print(f'{name}{tag}: {len(v2_revs)} v2 revisions')
    print(f'{"=" * 70}')

    for i, rev in enumerate(v2_revs):
        rev_id = rev.get('id', '?')
        mod_time = rev.get('modifiedDate', '?')[:19]
        user = rev.get('lastModifyingUser', {})
        author = user.get('displayName') or user.get('emailAddress', '?')
        size = rev.get('fileSize', '?')
        pinned = rev.get('pinned', False)
        pub = rev.get('published', False)
        pub_auto = rev.get('publishAuto', False)
        export = rev.get('exportLinks', {})

        flags = []
        if pinned:
            flags.append('PINNED')
        if pub:
            flags.append('PUB')
        if pub_auto:
            flags.append('AUTO')
        if export:
            flags.append(f'export:{len(export)}')

        flag_str = f'  [{" ".join(flags)}]' if flags else ''

        print(f'  {i+1:3d}. rev={str(rev_id):>6s}  {mod_time}  '
              f'{author:30s}  size={str(size):>10s}{flag_str}')

        # Show export link formats for first revision of each sheet
        if export and i == 0:
            print(f'       Export formats:')
            for fmt in sorted(export.keys()):
                short_fmt = fmt.split('/')[-1] if '/' in fmt else fmt
                # Show abbreviated URL to see revision ID pattern
                url = export[fmt]
                qs = url.split('?')[-1] if '?' in url else ''
                print(f'         {short_fmt}: ...?{qs[:80]}')

## Step 7: `files.get` — Version Field & File Metadata

A single lightweight API call per sheet. Returns the `version` counter —
yet another number representing internal edit state — plus `headRevisionId`,
`createdTime`, `owners`, and other metadata. We call both v3 and v2 since
v2 includes file-level `exportLinks`.

In [ ]:
files_get_v3 = {}
files_get_v2 = {}

print('=' * 70)
print('files.get (v3 + v2) — version, headRevisionId, metadata')
print('=' * 70)
print()

for s in all_sheets:
    fid = s.get('id', '?')
    name = s.get('name', '?')
    v3_count = s.get('revision_count', 0)
    known = s.get('known_as', '')
    tag = f' [{known}]' if known else ''

    if v3_count == 0:
        continue

    # --- v3 files.get ---
    url_v3 = (
        f'https://www.googleapis.com/drive/v3/files/{fid}'
        f'?fields=id,name,mimeType,version,modifiedTime,createdTime,'
        f'size,owners/displayName,owners/emailAddress,'
        f'lastModifyingUser/displayName,lastModifyingUser/emailAddress,'
        f'shared,webViewLink,headRevisionId'
    )
    data_v3 = api_get(url_v3, f'v3_files_get:{name}')
    files_get_v3[fid] = data_v3

    # --- v2 files.get ---
    url_v2 = f'https://www.googleapis.com/drive/v2/files/{fid}'
    data_v2 = api_get(url_v2, f'v2_files_get:{name}')
    files_get_v2[fid] = data_v2

    if data_v3:
        version = data_v3.get('version')
        created = data_v3.get('createdTime', '?')[:19]
        modified = data_v3.get('modifiedTime', '?')[:19]
        head_rev = data_v3.get('headRevisionId', '?')
        owners = [o.get('displayName', '?') for o in data_v3.get('owners', [])]
        last_user = data_v3.get('lastModifyingUser', {})
        last_mod_by = last_user.get('displayName') or last_user.get('emailAddress', '?')

        print(f'{name}{tag}')
        print(f'  v3 version:     {version}')
        print(f'  Created:        {created}')
        print(f'  Modified:       {modified}')
        print(f'  Head rev ID:    {head_rev}')
        print(f'  Owners:         {", ".join(owners)}')
        print(f'  Last mod by:    {last_mod_by}')
        print(f'  Shared:         {data_v3.get("shared")}')

    if data_v2:
        v2_version = data_v2.get('version')
        v2_export = data_v2.get('exportLinks', {})
        v2_title = data_v2.get('title', '?')
        print(f'  v2 version:     {v2_version}')
        if v2_export:
            export_fmts = [fmt.split('/')[-1] for fmt in sorted(v2_export.keys())]
            print(f'  v2 exportLinks: {", ".join(export_fmts)}')

    print()
    time.sleep(0.5)

## Step 8: Cross-Method Comparison

Compare all four independent metrics for each sheet:
- **v3 revisions** — number from Drive API v3 `revisions.list`
- **v2 revisions** — number from Drive API v2 `revisions.list`
- **version** — `files.get` version counter
- **max rev** — highest working internal revision via export URL

Also compares which revision IDs appear in v2 vs v3 to detect differences.

In [ ]:
# Build probe_results_by_id for quick lookup (probe_results is a list)
probe_by_id = {r['id']: r for r in probe_results}

# Also add known archived sheets' probe info
for fid, known_max in KNOWN_MAX_REVS.items():
    if fid not in probe_by_id:
        probe_by_id[fid] = {
            'id': fid,
            'internal_max_revision': known_max,
            'known_as': KNOWN_SHEETS.get(fid, ''),
        }

print('=' * 90)
print('CROSS-METHOD COMPARISON: All four revision metrics per sheet')
print('=' * 90)
print()
print(f'{"Sheet":40s} {"v3":>5s} {"v2":>5s} {"version":>10s} {"max_rev":>8s} '
      f'{"v3/max":>8s} {"density":>8s}')
print('-' * 90)

comparison_data = {}

for s in all_sheets:
    fid = s.get('id', '?')
    name = s.get('name', '?')
    v3_count = s.get('revision_count', 0)
    if v3_count == 0:
        continue

    v2_count = len(v2_results.get(fid, []))

    fg = files_get_v3.get(fid, {})
    version = int(fg.get('version', 0)) if fg else 0

    pr = probe_by_id.get(fid, {})
    max_rev = pr.get('internal_max_revision', 0)

    ratio = f'{v3_count/max_rev:.3f}' if max_rev > 0 else '?'
    density = f'{max_rev/v3_count:.0f}x' if v3_count > 0 and max_rev > 0 else '?'

    known = s.get('known_as', '')
    tag = f' [{known}]' if known else ''
    archived = fid in already_archived_ids

    comparison_data[fid] = {
        'name': name,
        'v3_revision_count': v3_count,
        'v2_revision_count': v2_count,
        'files_get_version': version,
        'internal_max_revision': max_rev,
        'known_as': known,
        'archived': archived,
    }

    print(f'{(name + tag):40s} {v3_count:5d} {v2_count:5d} '
          f'{version:10d} {max_rev:8d} {ratio:>8s} {density:>8s}')

print()
print('Legend:')
print('  v3       = Drive API v3 revisions.list count')
print('  v2       = Drive API v2 revisions.list count')
print('  version  = files.get version field')
print('  max_rev  = highest working export revision number')
print('  v3/max   = ratio of API revisions to internal revisions')
print('  density  = internal revisions per API revision (higher = more hidden edits)')

# --- v2 vs v3 revision ID comparison ---
print(f'\n\n{"=" * 70}')
print('v2 vs v3 REVISION ID COMPARISON')
print(f'{"=" * 70}')

for s in all_sheets:
    fid = s.get('id', '?')
    name = s.get('name', '?')
    v3_revs = s.get('revisions', [])
    v2_revs = v2_results.get(fid, [])

    if not v3_revs and not v2_revs:
        continue

    v3_ids = set(str(r.get('id')) for r in v3_revs)
    v2_ids = set(str(r.get('id')) for r in v2_revs)

    common = v3_ids & v2_ids
    only_v3 = v3_ids - v2_ids
    only_v2 = v2_ids - v3_ids

    known = s.get('known_as', '')
    tag = f' [{known}]' if known else ''

    print(f'\n{name}{tag}:')
    print(f'  v3: {len(v3_ids)} IDs, v2: {len(v2_ids)} IDs')
    print(f'  Common: {len(common)}, Only in v3: {len(only_v3)}, Only in v2: {len(only_v2)}')

    if only_v3:
        print(f'  IDs only in v3: {sorted(only_v3)[:10]}')
    if only_v2:
        print(f'  IDs only in v2: {sorted(only_v2)[:10]}')

    # Compare timestamps for common IDs
    if common:
        v3_by_id = {str(r.get('id')): r for r in v3_revs}
        v2_by_id = {str(r.get('id')): r for r in v2_revs}
        mismatches = []
        for rid in sorted(common):
            v3_time = v3_by_id[rid].get('modifiedTime', '?')
            v2_time = v2_by_id[rid].get('modifiedDate', '?')
            if v3_time != v2_time:
                mismatches.append((rid, v3_time, v2_time))
        if mismatches:
            print(f'  Timestamp mismatches: {len(mismatches)}')
            for rid, v3t, v2t in mismatches[:5]:
                print(f'    rev {rid}: v3={v3t[:19]}, v2={v2t[:19]}')
        else:
            print(f'  All {len(common)} common revisions have matching timestamps')

## Step 9: Revision Density & Editing Activity Deep Dive

For each sheet with significant history, combine all data sources to analyze:
- Monthly activity breakdown with author attribution
- Busiest editing days
- Largest gaps between consecutive API revisions
- Estimated number of hidden edits between API-visible revisions

In [ ]:
from datetime import datetime as dt

print('=' * 70)
print('REVISION DENSITY & EDITING ACTIVITY ANALYSIS')
print('=' * 70)

for s in all_sheets:
    fid = s.get('id', '?')
    name = s.get('name', '?')
    v3_revs = s.get('revisions', [])
    v2_revs = v2_results.get(fid, [])
    pr = probe_by_id.get(fid, {})
    max_rev = pr.get('internal_max_revision', 0)
    fg = files_get_v3.get(fid, {})
    version = int(fg.get('version', 0)) if fg else 0

    if not v3_revs:
        continue

    # Use whichever API gave more revisions for timeline analysis
    if len(v2_revs) > len(v3_revs):
        best_revs = v2_revs
        time_key = 'modifiedDate'
        api_label = 'v2'
    else:
        best_revs = v3_revs
        time_key = 'modifiedTime'
        api_label = 'v3'

    known = s.get('known_as', '')
    tag = f' [{known}]' if known else ''

    print(f'\n{"=" * 70}')
    print(f'{name}{tag}')
    print(f'{"=" * 70}')
    print(f'  API revisions (best: {api_label}): {len(best_revs)}')
    print(f'  Internal max revision:            {max_rev}')
    print(f'  files.get version:                {version}')

    if max_rev > 0 and len(best_revs) > 0:
        density = max_rev / len(best_revs)
        print(f'  Density: ~{density:.0f} internal revisions per API revision')

    # Monthly activity with authors
    months = Counter()
    authors_by_month = {}
    days = Counter()

    for rev in best_revs:
        mt = rev.get(time_key, '')
        if not mt or len(mt) < 10:
            continue
        day = mt[:10]
        month = mt[:7]
        days[day] += 1
        months[month] += 1

        user = rev.get('lastModifyingUser', {})
        author = user.get('displayName') or user.get('emailAddress', '?')
        if month not in authors_by_month:
            authors_by_month[month] = Counter()
        authors_by_month[month][author] += 1

    if months:
        print(f'\n  Monthly activity with authors (using {api_label} data):')
        for month in sorted(months.keys()):
            count = months[month]
            bar = '#' * min(count, 40)
            auth_summary = ', '.join(
                f'{a}({c})' for a, c in authors_by_month[month].most_common(3)
            )
            print(f'    {month}: {count:3d} {bar}  [{auth_summary}]')

    if days:
        print(f'\n  Busiest days:')
        for day, count in days.most_common(10):
            print(f'    {day}: {count} revisions')

    # Gap analysis
    if len(best_revs) > 1:
        gaps = []
        for i in range(1, len(best_revs)):
            t1 = best_revs[i-1].get(time_key, '')
            t2 = best_revs[i].get(time_key, '')
            if t1 and t2:
                try:
                    d1 = dt.fromisoformat(t1.replace('Z', '+00:00'))
                    d2 = dt.fromisoformat(t2.replace('Z', '+00:00'))
                    gap = (d2 - d1).total_seconds()
                    gaps.append((gap, i, t1, t2))
                except (ValueError, TypeError):
                    pass

        if gaps:
            gaps.sort(reverse=True)
            print(f'\n  Largest gaps between API revisions:')
            for gap_s, idx, t1, t2 in gaps[:5]:
                gap_days = gap_s / 86400
                print(f'    {gap_days:6.1f} days: rev {idx} -> {idx+1} '
                      f'({t1[:10]} -> {t2[:10]})')

## Step 10: Spot-Check — Sample Revision Content Download

For non-archived sheets with a meaningful internal revision count,
download a few sample revisions (first, 25%, 50%, 75%, last) to verify
the export URL returns genuinely different content at different revision
numbers. This confirms the revisions contain real changes.

In [ ]:
def download_revision_csv(sheet_id, rev_num):
    """Download a single revision as CSV, return (size, sha256) or None."""
    ensure_fresh_token()
    url = (
        f'https://docs.google.com/spreadsheets/export'
        f'?id={sheet_id}&revision={rev_num}&exportFormat=csv'
    )
    headers = {'Authorization': f'Bearer {TOKEN}'}
    try:
        req = Request(url, headers=headers)
        with urlopen(req, timeout=60) as resp:
            data = resp.read()
            h = hashlib.sha256(data).hexdigest()
            return len(data), h
    except Exception:
        return None


print('=' * 70)
print('SPOT CHECK: Sample revision downloads for non-archived sheets')
print('=' * 70)
print()

spot_check_results = {}

for s in all_sheets:
    fid = s.get('id', '?')
    name = s.get('name', '?')
    known = s.get('known_as', '')
    tag = f' [{known}]' if known else ''

    if fid in already_archived_ids:
        continue

    pr = probe_by_id.get(fid, {})
    max_rev = pr.get('internal_max_revision', 0)

    if max_rev == 0:
        continue

    # Sample: first, 25%, 50%, 75%, last
    sample_points = sorted(set([
        1,
        max(1, max_rev // 4),
        max(1, max_rev // 2),
        max(1, max_rev * 3 // 4),
        max_rev,
    ]))

    print(f'{name}{tag} (max_rev={max_rev}): sampling {len(sample_points)} points')

    hashes_seen = set()
    samples = []
    for rev_num in sample_points:
        result = download_revision_csv(fid, rev_num)
        if result:
            size, h = result
            is_new = h not in hashes_seen
            hashes_seen.add(h)
            tag_str = 'UNIQUE' if is_new else 'DUPLICATE'
            samples.append({
                'rev': rev_num, 'size': size,
                'hash': h, 'unique': is_new,
            })
            print(f'  rev {rev_num:6d}: {size:>10,} bytes  '
                  f'{h[:16]}...  [{tag_str}]')
        else:
            samples.append({'rev': rev_num, 'error': True})
            print(f'  rev {rev_num:6d}: FAILED')
        time.sleep(1)

    unique_count = len(hashes_seen)
    spot_check_results[fid] = {
        'name': name,
        'max_rev': max_rev,
        'samples': samples,
        'unique_count': unique_count,
        'total_sampled': len(sample_points),
    }
    print(f'  -> {unique_count}/{len(sample_points)} unique content states')
    if unique_count > 1:
        print(f'  -> CONFIRMED: revision history contains real changes')
    else:
        print(f'  -> WARNING: all samples identical — may be static')
    print()

## Step 11: Save All Results to Drive

Saves everything gathered from all four methods into a single JSON file.

In [ ]:
end_time = now_iso()

# Build the output with all method results
output = {
    '_provenance': {
        'description': 'Complete inventory of OASIS UBL TC Google Drive shared folder',
        'methods_used': [
            'Drive API v3 revisions.list (folder exploration)',
            'Binary search via export URL (internal max revision)',
            'Drive API v2 revisions.list (exportLinks, pinned, published)',
            'Drive API v3 files.get (version, headRevisionId)',
            'Drive API v2 files.get (exportLinks, additional metadata)',
            'Spot-check CSV downloads at sample revision points',
        ],
        'root_folder_url': f'https://drive.google.com/drive/folders/{ROOT_FOLDER_ID}',
        'root_folder_id': ROOT_FOLDER_ID,
        'started_at': start_time,
        'completed_at': end_time,
        'total_api_calls': len(api_log),
    },
    'stats': {
        'total_folders': folders,
        'total_files': files,
        'total_spreadsheets': sheets,
        'total_shortcuts': shortcuts,
        'total_api_revisions': total_revs,
    },
    'spreadsheet_summary': [
        {
            'name': s.get('name'),
            'id': s.get('id'),
            'path': s.get('path'),
            'revision_count': s.get('revision_count', 0),
            'first_revision': s.get('first_revision_time'),
            'last_revision': s.get('last_revision_time'),
            'known_as': s.get('known_as'),
            'via_shortcut': s.get('via_shortcut', False),
        }
        for s in all_sheets
    ],
    'probe_results': probe_results,
    'v2_revisions': {
        fid: revs for fid, revs in v2_results.items()
    },
    'files_get_v3': {
        fid: data for fid, data in files_get_v3.items() if data
    },
    'files_get_v2': {
        fid: data for fid, data in files_get_v2.items() if data
    },
    'comparison': comparison_data,
    'spot_checks': spot_check_results,
    'tree': tree,
    'api_log': api_log,
}

# Save to Drive
discovery_path = DRIVE_DIR / 'drive-discovery.json'
discovery_path.write_text(json.dumps(output, indent=2))
print(f'Saved: {discovery_path} ({discovery_path.stat().st_size:,} bytes)')

# Human-readable summary
summary_lines = []
summary_lines.append('OASIS UBL TC Google Drive - Complete Inventory')
summary_lines.append(f'Generated: {end_time}')
summary_lines.append(f'Root: https://drive.google.com/drive/folders/{ROOT_FOLDER_ID}')
summary_lines.append(f'Methods: v3 revisions, v2 revisions, files.get, binary search, spot-check')
summary_lines.append('')
summary_lines.append(f'Folders: {folders}, Files: {files}, Spreadsheets: {sheets}, Shortcuts: {shortcuts}')
summary_lines.append(f'Total API-listed revisions: {total_revs}')
summary_lines.append(f'Total API calls: {len(api_log)}')
summary_lines.append('')

summary_lines.append('CROSS-METHOD COMPARISON:')
summary_lines.append(f'{"Sheet":40s} {"v3":>5s} {"v2":>5s} {"version":>10s} {"max_rev":>8s}')
summary_lines.append('-' * 70)
for fid, c in comparison_data.items():
    name = c['name']
    known = c.get('known_as', '')
    tag = f' [{known}]' if known else ''
    summary_lines.append(
        f'{(name + tag):40s} {c["v3_revision_count"]:5d} '
        f'{c["v2_revision_count"]:5d} {c["files_get_version"]:10d} '
        f'{c["internal_max_revision"]:8d}'
    )

summary_lines.append('')
summary_lines.append('SPREADSHEETS BY REVISION COUNT:')
summary_lines.append('')

for i, s in enumerate(all_sheets, 1):
    fid = s.get('id', '?')
    name = s.get('name', '?')
    rc = s.get('revision_count', 0)
    known = s.get('known_as', '')
    tag = f' [{known}]' if known else ''
    first = (s.get('first_revision_time') or '?')[:10]
    last = (s.get('last_revision_time') or '?')[:10]
    summary_lines.append(f'{i:3d}. {name}: {rc} revisions ({first} -> {last}){tag}')
    summary_lines.append(f'     ID: {fid}')

if probe_results:
    summary_lines.append('')
    summary_lines.append('INTERNAL REVISION PROBES:')
    summary_lines.append('')
    for r in probe_results:
        tag = f' [{r["known_as"]}]' if r['known_as'] else ''
        summary_lines.append(
            f'  {r["name"]}: API={r["api_revision_count"]}, '
            f'Internal max={r["internal_max_revision"]}, '
            f'Ratio={r["ratio"]}x{tag}'
        )

if spot_check_results:
    summary_lines.append('')
    summary_lines.append('SPOT-CHECK RESULTS:')
    summary_lines.append('')
    for fid, sc in spot_check_results.items():
        summary_lines.append(
            f'  {sc["name"]}: {sc["unique_count"]}/{sc["total_sampled"]} '
            f'unique at sample points (max_rev={sc["max_rev"]})'
        )

if new_discoveries:
    summary_lines.append('')
    summary_lines.append('NEW DISCOVERIES (not previously known):')
    summary_lines.append('')
    for s in new_discoveries:
        summary_lines.append(f'  {s["name"]}: {s.get("revision_count", 0)} revisions')
        summary_lines.append(f'    ID: {s["id"]}')

summary_text = '\n'.join(summary_lines)
summary_path = DRIVE_DIR / 'drive-discovery-summary.txt'
summary_path.write_text(summary_text)
print(f'Saved: {summary_path} ({summary_path.stat().st_size:,} bytes)')

print(f'\n--- Summary ---\n')
print(summary_text)

## Step 12: Next Steps

After running this notebook, use the results to decide which sheets
need full revision-by-revision download.

### Decision criteria

1. **High internal max revision** (>100) with confirmed unique content
   at spot-check points — these sheets have substantial editing history
   worth archiving in full using `download-all-revisions.ipynb`.

2. **High density** (many internal revisions per API revision) — these
   sheets had frequent small edits that Google merged into fewer visible
   revisions. The editing history between API-visible revisions could
   contain interesting intermediate states.

3. **v2 exportLinks available** — if v2 provides revision-specific
   `exportLinks`, those may be an alternative download method that
   returns content at specific API-visible revision points.

### What to look for

- **UBL 2.4 Library/Documents**: Most likely candidates for significant
  revision history. If max_rev >100, they contain 2.4 development
  history across CSD01 through OS stages.

- **UBL 2.5 Signature**: Small sheet, likely few meaningful revisions.
  Worth confirming with the spot-check results.

- **Unknown sheets**: Any newly discovered spreadsheet with significant
  revision history is a valuable find.

### Files on Drive

```
Drive: ubl-gc-revisions/
├── drive-discovery.json          ← all data from all methods
└── drive-discovery-summary.txt   ← human-readable summary
```